# Lunyoro-Rutooro Translator — Colab Backend

Runs the FastAPI backend with NLLB-200 FP32 on a free Colab GPU.
Models are loaded directly from HuggingFace Hub at startup.
The public URL is obtained via **Cloudflare Tunnel** (built into Colab — no account needed).

## Architecture
```
Browser / Phone
    └─ Vercel (horizonx.kathay.tech)
           NEXT_PUBLIC_API_URL = <Colab public URL>
                └─ Cloudflare Tunnel (auto, no sign-up)
                       └─ FastAPI on port 7860 (this notebook)
                              ├─ NLLB-200 FP32  ← loaded from HF Hub
                              ├─ MobileNetV2     ← loaded from HF Hub
                              └─ EasyOCR
```

## Architecture
```
Browser → Vercel (horizonx.kathay.tech)
             NEXT_PUBLIC_API_URL = Cloudflare Worker (never changes)
                  └─ Worker probes Colab URL first, falls back to HF Space
                       └─ Colab T4 (this notebook) ← NLLB loaded from HF Hub
```

## Steps
1. `Runtime → Change runtime type → T4 GPU`
2. Fill in `HF_TOKEN` and `WORKER_SECRET` in **Cell 1**
3. `Runtime → Run all` — waits ~3 min for models to load
4. Cell 7 auto-POSTs this session's tunnel URL to the Cloudflare Worker
5. **Done** — Vercel immediately routes to this Colab T4, no redeploy needed

> Each new Colab session re-registers itself automatically.
> When the session ends, the Worker falls back to the HF Space.


In [ ]:
# ── Cell 1 — SECRETS ──────────────────────────────────────────────────────────
# HuggingFace token — needed to load the fine-tuned NLLB models.
# Get it at: https://huggingface.co/settings/tokens
HF_TOKEN = ""   # paste your HF token here

# Cloudflare Worker shared secret — must match the WORKER_SECRET you set with:
#   npx wrangler secret put WORKER_SECRET   (run once in cloudflare-worker/)
# This allows the notebook to register its tunnel URL with the Worker.
# Leave empty only if you skipped setting WORKER_SECRET on the Worker side.
WORKER_SECRET = ""   # e.g. "my-secret-token-123"

# HuggingFace model repos (fine-tuned NLLB-200 for Runyoro-Rutooro)
NLLB_EN2LUN_REPO = "keithtwesigye/lunyoro-nllb-en2lun"
NLLB_LUN2EN_REPO = "keithtwesigye/lunyoro-nllb-lun2en"

# LLM used for translation refinement via HF Inference Router
HF_CHAT_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

In [ ]:
# ── Cell 2 — INSTALL DEPENDENCIES ─────────────────────────────────────────────
!pip install -q \
    fastapi "uvicorn[standard]" \
    "transformers==4.51.3" sentencepiece accelerate \
    easyocr opencv-python-headless pillow \
    openai rapidfuzz python-multipart \
    PyPDF2 python-docx pdfplumber
print("✓ Dependencies installed")

In [ ]:
# ── Cell 3 — CHECK GPU ────────────────────────────────────────────────────────
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ No GPU — translations will be slow.")
    print("  Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── Cell 4 — LOAD NLLB MODELS FROM HF HUB ────────────────────────────────────
import os
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_CHAT_MODEL"] = HF_CHAT_MODEL

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading NLLB en→lun …")
tok_en2lun = AutoTokenizer.from_pretrained(
    NLLB_EN2LUN_REPO, src_lang="eng_Latn", token=HF_TOKEN)
mdl_en2lun = AutoModelForSeq2SeqLM.from_pretrained(
    NLLB_EN2LUN_REPO, token=HF_TOKEN).to(device)
mdl_en2lun.eval()
print("  ✓ en→lun ready")

print("Loading NLLB lun→en …")
tok_lun2en = AutoTokenizer.from_pretrained(
    NLLB_LUN2EN_REPO, src_lang="run_Latn", token=HF_TOKEN)
mdl_lun2en = AutoModelForSeq2SeqLM.from_pretrained(
    NLLB_LUN2EN_REPO, token=HF_TOKEN).to(device)
mdl_lun2en.eval()
print("  ✓ lun→en ready")

# run_Latn (Rundi proxy, ID 256146) works best for quantized/FP32 NLLB on this model.
# nyo_Latn (custom token ID 256205) embedding is distorted — outputs Kirundi.
NLLB_LANG_LUN = "run_Latn"
NLLB_LANG_EN  = "eng_Latn"
TGT_LUN_ID = tok_en2lun.convert_tokens_to_ids(NLLB_LANG_LUN)
TGT_EN_ID  = tok_lun2en.convert_tokens_to_ids(NLLB_LANG_EN)
print(f"  Token IDs — run_Latn:{TGT_LUN_ID}  eng_Latn:{TGT_EN_ID}")

In [ ]:
# ── Cell 5 — TRANSLATION HELPERS + MOBILENET + EASYOCR ───────────────────────
import re, io

# ── Sentence normalisation ────────────────────────────────────────────────────
def _sentence_case(text: str) -> str:
    """Capitalise first letter and add terminal punctuation — NLLB needs this."""
    text = text.strip()
    if text and not text[0].isupper():
        text = text[0].upper() + text[1:]
    if text and text[-1] not in ".!?,;:":
        qwords = ("how","what","where","when","who","why","which",
                  "is ","are ","do ","does ","can ","will ","have ")
        if any(text.lower().startswith(w) for w in qwords):
            text += "?"
        elif not text[-1].isdigit():
            text += "."
    return text

# ── Core translation ──────────────────────────────────────────────────────────
def translate_en2lun(text: str) -> str:
    text = _sentence_case(text)
    tok_en2lun.src_lang = NLLB_LANG_EN
    inputs = tok_en2lun(text, return_tensors="pt",
                         truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = mdl_en2lun.generate(
            **inputs, forced_bos_token_id=TGT_LUN_ID,
            num_beams=5, max_length=256,
            no_repeat_ngram_size=3, repetition_penalty=1.3,
        )
    result = tok_en2lun.decode(out[0], skip_special_tokens=True)
    result = re.sub(r"▁", " ", result).strip()
    result = re.sub(r"^(run_Latn|eng_Latn|nyk_Latn)\s*:\s*", "", result).strip()
    return result

def translate_lun2en(text: str) -> str:
    tok_lun2en.src_lang = NLLB_LANG_LUN
    inputs = tok_lun2en(text, return_tensors="pt",
                         truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = mdl_lun2en.generate(
            **inputs, forced_bos_token_id=TGT_EN_ID,
            num_beams=5, max_length=256,
            no_repeat_ngram_size=3, repetition_penalty=1.3,
        )
    return tok_lun2en.decode(out[0], skip_special_tokens=True).strip()

def _make_translation_response(text: str, direction: str) -> dict:
    nllb = translate_en2lun(text) if direction == "en->lun" else translate_lun2en(text)
    return {
        "translation": nllb,
        "translation_nllb": nllb,
        "translation_marian": None,
        "method": "neural_mt",
        "confidence": 0.85,
    }

# ── Qwen refinement (optional — only runs when HF_TOKEN is set) ───────────────
def _qwen_refine(source_en: str, draft_lun: str) -> str:
    """Refine a Runyoro draft via HF Inference Router. Returns draft on failure."""
    if not HF_TOKEN:
        return draft_lun
    try:
        from openai import OpenAI
        client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=HF_TOKEN)
        resp = client.chat.completions.create(
            model=HF_CHAT_MODEL,
            messages=[
                {"role": "system", "content": (
                    "You are a Runyoro-Rutooro grammar expert. "
                    "Correct the draft for noun class, R/L rule, and phrasing. "
                    "Output ONLY the corrected Runyoro-Rutooro text — no explanation."
                )},
                {"role": "user",
                 "content": f"English: {source_en}\nDraft: {draft_lun}\nCorrected:"},
            ],
            max_tokens=128, temperature=0.1,
        )
        refined = resp.choices[0].message.content.strip()
        # Only accept if it looks like a clean translation (no meta-commentary)
        first_line = refined.split("\n")[0].strip()
        if first_line and "(" not in first_line and len(first_line) < 200:
            return first_line
    except Exception:
        pass
    return draft_lun

# ── MobileNetV2 (Lens Identify) ───────────────────────────────────────────────
from transformers import MobileNetV2ForImageClassification, MobileNetV2ImageProcessor
print("Loading MobileNetV2 …")
mn_processor = MobileNetV2ImageProcessor.from_pretrained("google/mobilenet_v2_1.0_224")
mn_model     = MobileNetV2ForImageClassification.from_pretrained("google/mobilenet_v2_1.0_224")
mn_model.eval()
print("  ✓ MobileNetV2 ready")

# ── EasyOCR (Lens OCR) ────────────────────────────────────────────────────────
import easyocr
print("Loading EasyOCR …")
ocr_reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())
print("  ✓ EasyOCR ready")

# Smoke test
print("\nSmoke test:")
print(f"  en→lun: 'Good morning' → '{translate_en2lun('Good morning')}'")
print(f"  lun→en: 'Webale muno'  → '{translate_lun2en('Webale muno')}'")

In [ ]:
# ── Cell 6 — FASTAPI APP (all 17 endpoints) ──────────────────────────────────
# NOTE: Cell 7 (server + tunnel) depends on 'app' defined here.
# Always run cells in order: 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8
# Or use Runtime → Run all
#
assert 'translate_en2lun' in dir(), "Run Cell 5 first — translation helpers not loaded"

import base64, json, time, numpy as np
from datetime import datetime
from typing import Optional, List
from PIL import Image
from fastapi import FastAPI, HTTPException, UploadFile, File, Request
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="Lunyoro Translator — Colab")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Request models ────────────────────────────────────────────────────────────
class TranslateReq(BaseModel):
    text: str
    context: str = ""
    direction: str = "en->lun"
    refine: bool = False

class BatchReq(BaseModel):
    sentences: List[str]
    direction: str = "en->lun"

class LookupReq(BaseModel):
    word: str
    direction: str = "en→lun"

class SpellReq(BaseModel):
    text: str

class FeedbackReq(BaseModel):
    source_text: str
    translation: str
    direction: str = "en→lun"
    rating: int = 0
    correction: str = ""
    error_type: str = ""
    model_used: str = ""
    domain: str = ""

class ChatReq(BaseModel):
    message: str
    history: list = []

_history_store: list = []

# ── Health / meta ─────────────────────────────────────────────────────────────
@app.get("/")
def root():
    return {"status": "ok", "backend": "colab", "device": device}

@app.get("/health")
def health():
    return {"status": "ok", "timestamp": datetime.utcnow().isoformat()}

@app.get("/ping")
def ping():
    return {"pong": True}

@app.get("/system-info")
def system_info():
    return {
        "nllb_en2lun": True, "nllb_lun2en": True,
        "nllb_en2lun_format": f"fp32_{device}",
        "nllb_lun2en_format": f"fp32_{device}",
        "marian_en2lun": False, "marian_lun2en": False,
        "device": device, "backend": "colab",
    }

@app.get("/config")
def config():
    # No ngrok / Pi — just confirm this is the Colab backend
    return {"backend": "colab", "ngrok_url": None, "probe_url": None}

# ── Translation ───────────────────────────────────────────────────────────────
@app.post("/translate")
def translate_text(req: TranslateReq):
    if not req.text.strip():
        raise HTTPException(400, "Text cannot be empty")
    r = _make_translation_response(req.text, "en->lun")
    if req.refine and r["translation"]:
        refined = _qwen_refine(req.text, r["translation"])
        r["translation"] = r["translation_nllb"] = refined
    _history_store.insert(0, {"input": req.text, "direction": "en→lun",
                               "translation": r["translation"],
                               "timestamp": datetime.utcnow().isoformat()})
    return r

@app.post("/translate-reverse")
def translate_reverse(req: TranslateReq):
    if not req.text.strip():
        raise HTTPException(400, "Text cannot be empty")
    r = _make_translation_response(req.text, "lun->en")
    _history_store.insert(0, {"input": req.text, "direction": "lun→en",
                               "translation": r["translation"],
                               "timestamp": datetime.utcnow().isoformat()})
    return r

@app.post("/translate-batch")
def translate_batch(req: BatchReq):
    if not req.sentences:
        raise HTTPException(400, "No sentences")
    results = []
    for s in req.sentences[:100]:
        r = _make_translation_response(s, req.direction)
        results.append({"source": s, "translation": r["translation"],
                         "method": r["method"], "confidence": r["confidence"]})
    return {"results": results, "total": len(results), "direction": req.direction}

@app.post("/translate-batch-file")
async def translate_batch_file(file: UploadFile = File(...), direction: str = "en->lun"):
    import csv
    contents = await file.read()
    text = contents.decode("utf-8", errors="ignore")
    if file.filename and file.filename.endswith(".csv"):
        sentences = [row[0].strip() for row in csv.reader(text.splitlines()) if row]
    else:
        sentences = [l.strip() for l in text.splitlines() if l.strip()]
    results = []
    for s in sentences[:200]:
        r = _make_translation_response(s, direction)
        results.append({"source": s, "translation": r["translation"], "method": r["method"]})
    return {"results": results, "total": len(results),
            "direction": direction, "filename": file.filename}

# ── Lens: Identify (classify-image) ──────────────────────────────────────────
@app.post("/classify-image")
async def classify_image(file: UploadFile = File(...), top_k: int = 1):
    contents = await file.read()
    try:
        img = Image.open(io.BytesIO(contents)).convert("RGB")
    except Exception as e:
        raise HTTPException(400, f"Cannot decode image: {e}")
    inputs = mn_processor(images=img, return_tensors="pt")
    with torch.no_grad():
        logits = mn_model(**inputs).logits
    probs = torch.nn.functional.softmax(logits, dim=-1)[0]
    top_probs, top_idx = torch.topk(probs, min(top_k, 5))
    results = []
    for prob, idx in zip(top_probs, top_idx):
        label_en = mn_model.config.id2label[idx.item()].split(",")[0].strip().lower()
        label_lun = translate_en2lun(label_en)
        refined   = _qwen_refine(label_en, label_lun)
        results.append({
            "label_en": label_en,
            "label_lun": refined,
            "label_lun_raw": label_lun,
            "confidence": round(prob.item(), 4),
            "method": "nllb+qwen" if HF_TOKEN else "nllb",
        })
    return {"predictions": results, "top_k": len(results),
            "model": "google/mobilenet_v2_1.0_224"}

@app.get("/classify-image/status")
def classify_status():
    return {"ready": True, "error": None}

# ── Lens: OCR ─────────────────────────────────────────────────────────────────
def _ocr_and_translate(img_bytes: bytes, direction: str) -> dict:
    import cv2
    arr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None:
        return {"error": "Cannot decode image", "regions": []}
    h, w = img.shape[:2]
    raw = ocr_reader.readtext(img)
    regions = []
    for (bbox, text, conf) in raw:
        if conf < 0.3 or not text.strip():
            continue
        translated = _make_translation_response(text, direction)["translation"]
        xs = [p[0] for p in bbox]; ys = [p[1] for p in bbox]
        x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
        regions.append({
            "original": text, "translated": translated,
            "confidence": round(conf, 2),
            "bbox": {"x": int(x1), "y": int(y1),
                     "width": int(x2-x1), "height": int(y2-y1)},
            "bbox_norm": {"x": round(x1/w, 4), "y": round(y1/h, 4),
                          "width": round((x2-x1)/w, 4),
                          "height": round((y2-y1)/h, 4)},
        })
    return {"regions": regions, "total_detected": len(raw),
            "total_translated": len(regions),
            "image_size": {"width": w, "height": h}, "direction": direction}

@app.post("/ocr-translate")
async def ocr_translate(file: UploadFile = File(...), direction: str = "en->lun"):
    return _ocr_and_translate(await file.read(), direction)

@app.post("/ocr-translate-base64")
async def ocr_translate_b64(request: Request):
    body = await request.json()
    img_data = body.get("image", "")
    direction = body.get("direction", "en->lun")
    if "," in img_data:
        img_data = img_data.split(",", 1)[1]
    try:
        img_bytes = base64.b64decode(img_data)
    except Exception as e:
        return {"error": f"Bad base64: {e}", "regions": []}
    return _ocr_and_translate(img_bytes, direction)

# ── Chat (HF Router → translated to Runyoro) ──────────────────────────────────
@app.post("/chat")
async def chat(req: ChatReq):
    SYSTEM = (
        "You are a helpful assistant for the Runyoro-Rutooro language of Uganda. "
        "Reply in clear, simple English (2-4 sentences). "
        "Your reply will be translated into Runyoro-Rutooro by a separate model — "
        "do NOT include any Runyoro words yourself."
    )
    english = ""
    if HF_TOKEN:
        try:
            from openai import OpenAI
            client = OpenAI(base_url="https://router.huggingface.co/v1",
                            api_key=HF_TOKEN)
            msgs = [{"role": "system", "content": SYSTEM}]
            for h in req.history[-4:]:
                msgs.append({"role": h.get("role", "user"),
                              "content": h.get("content", "")})
            msgs.append({"role": "user", "content": req.message})
            resp = client.chat.completions.create(
                model=HF_CHAT_MODEL, messages=msgs,
                max_tokens=256, temperature=0.7,
            )
            english = resp.choices[0].message.content.strip()
        except Exception:
            pass
    if not english:
        english = (
            "I can help you learn Runyoro-Rutooro. "
            "Try asking me to translate a phrase, or ask about greetings or grammar."
        )
    # Translate English reply sentence-by-sentence
    sentences = [s.strip()
                 for s in re.split(r"(?<=[.!?])\s+", english) if s.strip()]
    runyoro = " ".join(translate_en2lun(s) for s in sentences)
    return {
        "reply": f"{runyoro}\n\n(English: {english})",
        "reply_nllb": runyoro,
        "reply_marian": None,
    }

# ── Dictionary / language rules (lightweight stubs) ──────────────────────────
@app.post("/lookup")
def lookup(req: LookupReq):
    direction = "en->lun" if "en" in req.direction.lower() else "lun->en"
    r = _make_translation_response(req.word, direction)
    return {"word": req.word,
            "results": [{"translation": r["translation"], "method": "nllb"}]}

@app.post("/spellcheck")
def spellcheck(req: SpellReq):
    return {"misspelled": []}  # no corpus on Colab

@app.get("/language-rules/interjections")
def interjections():
    return {"interjections": {
        "Agandi": "How are you?",
        "Oraire ota": "Good morning",
        "Webale muno": "Thank you very much",
        "Mpora": "Take it easy / Slowly",
        "Genda kurungi": "Goodbye / Go well",
    }}

@app.get("/language-rules/idioms")
def idioms():
    return {"idioms": {}}

# ── Feedback / history ────────────────────────────────────────────────────────
@app.post("/feedback")
def feedback(req: FeedbackReq):
    return {"status": "saved"}

@app.get("/history")
def history():
    return {"history": _history_store[:100]}

# ── PDF summarise (not available on Colab) ────────────────────────────────────
@app.post("/summarize-pdf")
async def summarize_pdf(file: UploadFile = File(...)):
    raise HTTPException(501, "PDF summarization is not available in the Colab backend.")

print("✓ FastAPI app ready — all endpoints defined")

In [ ]:
# ── Cell 7 — START SERVER + TUNNEL + AUTO-REGISTER WITH CLOUDFLARE WORKER ────
#
# Depends on: Cell 1 (HF_TOKEN, WORKER_SECRET), Cell 6 (app)
# 1. Starts uvicorn (FastAPI on port 7860)
# 2. Opens a cloudflared tunnel → *.trycloudflare.com
# 3. POSTs the tunnel URL to the Cloudflare Worker so Vercel routes here

import subprocess, threading, time, re as _re, requests

# Guard — make sure Cell 6 ran first
assert 'app' in dir(), "Run Cell 6 first — FastAPI app not defined"

PORT = 7860
WORKER_URL = "https://lunyoro-translator-api.keithtwesigye74.workers.dev"
# WORKER_SECRET is read from Cell 1 — do NOT redefine it here

# ── Start uvicorn ─────────────────────────────────────────────────────────────
def _run_uvicorn():
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

threading.Thread(target=_run_uvicorn, daemon=True).start()
time.sleep(3)

try:
    r = requests.get(f"http://localhost:{PORT}/health", timeout=5)
    assert r.status_code == 200
    print(f"✓ uvicorn running on port {PORT}")
except Exception as e:
    print(f"✗ uvicorn not responding: {e}")
    raise

# ── Install cloudflared ───────────────────────────────────────────────────────
import subprocess, os
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("Installing cloudflared …")
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared"
    ], check=True)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
    print("✓ cloudflared installed")

# ── Start tunnel ──────────────────────────────────────────────────────────────
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
print("Waiting for cloudflared tunnel …", end="", flush=True)
for _ in range(60):
    line = cf_proc.stdout.readline()
    m = _re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break
    print(".", end="", flush=True)
print()

if not public_url:
    print("✗ Could not get tunnel URL. Try: !cloudflared tunnel --url http://localhost:7860")
    raise RuntimeError("cloudflared tunnel failed")

print(f"✓ Tunnel live: {public_url}")

# ── Register URL with Cloudflare Worker ───────────────────────────────────────
print(f"\nRegistering with Worker at {WORKER_URL} …")
try:
    headers = {"Content-Type": "application/json"}
    if WORKER_SECRET:
        headers["X-Worker-Secret"] = WORKER_SECRET
    resp = requests.post(
        f"{WORKER_URL}/update-colab-url",
        json={"url": public_url},
        headers=headers,
        timeout=15,
    )
    if resp.status_code == 200:
        print(f"✓ Worker updated — Vercel will now route to this Colab session")
    elif resp.status_code == 401:
        print(f"⚠ Worker rejected the request (401) — set WORKER_SECRET to match the Worker secret")
        print(f"  Manual fallback: set NEXT_PUBLIC_API_URL={public_url} in Vercel")
    else:
        print(f"⚠ Worker returned {resp.status_code}: {resp.text[:120]}")
        print(f"  Manual fallback: set NEXT_PUBLIC_API_URL={public_url} in Vercel")
except Exception as e:
    print(f"⚠ Could not reach Worker: {e}")
    print(f"  Manual fallback: set NEXT_PUBLIC_API_URL={public_url} in Vercel")

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("=" * 62)
print(f"  Colab tunnel  : {public_url}")
print(f"  Worker URL    : {WORKER_URL}  ← Vercel points here (never changes)")
print(f"  GPU           : {device}")
print("=" * 62)
print()
print("Vercel frontend (horizonx.kathay.tech) is now using this Colab T4 GPU.")
print("Keep this notebook running. The tunnel closes when the runtime stops.")

In [ ]:
# ── Cell 8 — VERIFY END-TO-END ────────────────────────────────────────────────
import requests

# Use localhost for the verification (avoids going out through the tunnel)
BASE = f"http://localhost:{PORT}"

h  = requests.get(f"{BASE}/health", timeout=5).json()
si = requests.get(f"{BASE}/system-info", timeout=5).json()
t1 = requests.post(f"{BASE}/translate",
     json={"text": "How are you?", "direction": "en->lun"}, timeout=30).json()
t2 = requests.post(f"{BASE}/translate-reverse",
     json={"text": "Webale muno", "direction": "lun->en"}, timeout=30).json()
t3 = requests.post(f"{BASE}/translate",
     json={"text": "Good morning, my friend.", "direction": "en->lun"}, timeout=30).json()

print("Health:     ", h)
print("Device:     ", si["device"])
print("en→lun 1:   'How are you?'          →", t1["translation"])
print("lun→en:     'Webale muno'            →", t2["translation"])
print("en→lun 2:   'Good morning, my friend.' →", t3["translation"])
print()
print("✓ All checks passed — share the URL above with Vercel.")